# Sentence Embeddings with Sentence-Transformers
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/06_GenAI_LLM_RAG/sentence_transformers_embeddings.ipynb)

Word2Vec/GloVe produce one vector per WORD. Sentence-transformers (SBERT) embed whole sentences so that meaning-similar texts land close together - the foundation of semantic search and RAG.

Model: `all-MiniLM-L6-v2` (80 MB, free download, runs on Colab CPU/GPU).

In [ ]:
!pip install -q sentence-transformers

## 1. Encode sentences

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
sentences = [
    "The bank approved my home loan quickly.",
    "I sat by the river bank watching ducks.",
    "Interest rates for mortgages are falling.",
    "A fox sprinted across the meadow.",
]
emb = model.encode(sentences)
print(emb.shape)          # 4 sentences x 384 dims

## 2. Similarity matrix - watch word-overlap lose to meaning

In [ ]:
import matplotlib.pyplot as plt
sims = util.cos_sim(emb, emb)

plt.imshow(sims, cmap="Blues", vmin=0, vmax=1)
for i in range(len(sentences)):
    for j in range(len(sentences)):
        plt.text(j, i, f"{sims[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.xticks(range(len(sentences)), ["loan", "river", "mortgage", "fox"], rotation=30)
plt.yticks(range(len(sentences)), ["loan", "river", "mortgage", "fox"])
plt.title("Cosine similarity"); plt.colorbar(); plt.show()

Rows 0 and 2 share NO keywords yet score high - both talk about loans. Word overlap (row 0 vs row 1: 'bank') is misleading; embeddings capture semantics.

## 3. Mini semantic search engine

In [ ]:
faq = {
    "How do I reset my password?": "Go to Settings > Security > Reset password.",
    "What is the refund policy?":  "Full refund within 30 days of purchase.",
    "Do you support two-factor authentication?": "Yes - SMS or authenticator apps.",
}
q_emb = model.encode(list(faq.keys()) + ["I forgot my login credentials"])
scores = util.cos_sim(q_emb[-1], q_emb[:-1])[0]
best = scores.argmax()
print(f"Q: I forgot my login credentials")
print(f"matched FAQ : {list(faq)[best]}  ({scores[best]:.2f})")
print(f"answer      : {list(faq.values())[best]}")

## Takeaways
- One model, one `encode()` call - no vocabulary/OOV headaches like Word2Vec.
- Cosine similarity + a matrix of vectors = search, clustering, dedup, RAG chunk-matching.
- For production scale, store these vectors in FAISS/Chroma -> next notebook.